In [1]:
import os
import glob
import pydicom
import xml.etree.ElementTree as ET
import numpy as np
import cv2
import random
import os
from pathlib import Path
from collections import defaultdict, Counter
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2


In [ ]:
DICOM_ROOT = "D:/DIno-LungDet-Data/Lung-PET-CT-Dx/"   # root folder with DICOM files
XML_ROOT = "D:/DIno-LungDet-Data/Annotation/"  # root folder with XML annotations
EXPORT_BASE = os.path.join("C:/Users/chris/Documents/GitHub/DINO-LungDet", "data", "processed") 

print(f"Suche DICOMs in:  {DICOM_ROOT}")
print(f"Suche XMLs in:    {XML_ROOT}")
print(f"Suche Export-Pfad in:    {EXPORT_BASE}")

# Output root

# Basic sanity check: does the annotation folder exist?
if os.path.exists(XML_ROOT):
    print(f"OK: Der XML Pfad wurde gefunden!")
else:
    print(f"FEHLER: Der Pfad wurde nicht gefunden!")
    print(f"Bitte prüfe, ob der Ordner existiert: {XML_ROOT}")

if os.path.exists(DICOM_ROOT):
    print(f"OK: Der DICOM Pfad wurde gefunden!")
else:
    print(f"FEHLER: Der Pfad wurde nicht gefunden!")
    print(f"Bitte prüfe, ob der Ordner existiert: {DICOM_ROOT}")

Suche DICOMs in:  D:/DIno-LungDet-Data/Lung-PET-CT-Dx/
Suche XMLs in:    D:/DIno-LungDet-Data/Annotation/
Suche Export-Pfad in:    C:/Users/chris/Documents/GitHub/DINO-LungDet\data\processed
✅ OK: Der XML Pfad wurde gefunden!
✅ OK: Der DICOM Pfad wurde gefunden!


In [3]:
# ----------------------------
# Dataset split configuration
# ----------------------------

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Create output folders for each split:
# data/processed/{train,val,test}/{images,labels}
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(EXPORT_BASE, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(EXPORT_BASE, split, 'labels'), exist_ok=True)

In [4]:
def get_patient_bit_profile(patient_id):
    """
    Returns one of:
    - only16
    - only8
    - both
    - zero
    - other
    """
    p_dicom_search = glob.glob(os.path.join(DICOM_ROOT, f"*{patient_id}*"))
    if not p_dicom_search:
        return "zero"

    all_dicoms = glob.glob(os.path.join(p_dicom_search[0], "**", "*.dcm"), recursive=True)

    n8, n16, no = 0, 0, 0

    p_xml_dir = os.path.join(XML_ROOT, patient_id)
    if not os.path.isdir(p_xml_dir):
        return "zero"

    xml_list = {
        f.replace(".xml", "")
        for f in os.listdir(p_xml_dir)
        if f.endswith(".xml")
    }

    for d_path in all_dicoms:
        try:
            ds_meta = pydicom.dcmread(d_path, stop_before_pixels=True)
            uid = ds_meta.SOPInstanceUID

            # nur annotierte/exportierte Bilder berücksichtigen
            if uid not in xml_list:
                continue

            ds = pydicom.dcmread(d_path, stop_before_pixels=True)
            bits = getattr(ds, "BitsAllocated", None)

            if bits == 8:
                n8 += 1
            elif bits == 16:
                n16 += 1
            else:
                no += 1
        except:
            continue

    if n8 == 0 and n16 == 0 and no == 0:
        return "zero"
    if n8 > 0 and n16 > 0:
        return "both"
    if n8 > 0 and n16 == 0 and no == 0:
        return "only8"
    if n16 > 0 and n8 == 0 and no == 0:
        return "only16"
    return "other"

In [50]:
def build_patient_records(all_patients):
    records = []

    for p in all_patients:
        prefix = p[0] if len(p) > 0 else "UNK"
        bit_profile = get_patient_bit_profile(p)

        # kombinierte Stratify-Klasse
        stratify_label = f"{prefix}_{bit_profile}"

        records.append({
            "patient_id": p,
            "prefix": prefix,
            "bit_profile": bit_profile,
            "stratify_label": stratify_label
        })

    return records

In [51]:
all_patients = sorted([
    d for d in os.listdir(XML_ROOT)
    if os.path.isdir(os.path.join(XML_ROOT, d))
])

records=build_patient_records(all_patients)
records

[{'patient_id': 'A0001',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0002',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0003',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0004',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0005',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0006',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0007',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0008',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0009',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_label': 'A_only16'},
 {'patient_id': 'A0010',
  'prefix': 'A',
  'bit_profile': 'only16',
  'stratify_l

In [52]:
Counter(r["stratify_label"] for r in records)

Counter({'A_only16': 154,
         'A_both': 94,
         'G_only16': 32,
         'B_only16': 29,
         'G_both': 29,
         'B_both': 9,
         'A_zero': 5,
         'E_only16': 5})

In [ ]:
def constrained_group_split(records, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO, random_state=42):
    """
    Split patients into train/val/test with constraints.

    Goal:
    - keep split approximately at 70/15/15
    - ensure that every group with >= 3 patients has
      at least 1 patient in val and 1 patient in test

    Parameters
    ----------
    records : list of dict
        Each dict must contain at least:
        - "patient_id"
        - "stratify_label"

        Example:
        {
            "patient_id": "A0171",
            "stratify_label": "A_both"
        }

    Returns
    -------
    train_patients, val_patients, test_patients : list
        Lists of patient IDs
    """

    # Safety check: ratios should sum to 1.0
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8

    rng = random.Random(random_state)

    total_n = len(records)

    # Target sizes for the three splits
    target_train = round(total_n * train_ratio)
    target_val = round(total_n * val_ratio)
    target_test = total_n - target_train - target_val

    # --------------------------------------------------
    # 1) Group patients by their stratify label
    # --------------------------------------------------
    groups = defaultdict(list)
    print(groups)
    for r in records:
        groups[r["stratify_label"]].append(r["patient_id"])
    print(groups)
    # Shuffle within each group so assignment is reproducible but random
    for label in groups:
        rng.shuffle(groups[label])

    train_patients = []
    val_patients = []
    test_patients = []

    leftovers = []

    # --------------------------------------------------
    # 2) Reserve at least:
    #    - 1 patient for val
    #    - 1 patient for test
    # for every group with >= 3 patients
    #
    # Why >= 3?
    # Because then we can still leave at least one patient
    # available for train or later balancing.
    # --------------------------------------------------
    for label, patients in groups.items():
        n = len(patients)

        if n >= 3:
            # Put one into val
            val_patients.append(patients[0])

            # Put one into test
            test_patients.append(patients[1])

            # Remaining patients go to leftovers for later assignment
            leftovers.extend(patients[2:])

        elif n <= 2:
            # With only 1 patient, we cannot place it in both val and test.
            # We keep it for later allocation.
            leftovers.extend(patients)

    # Shuffle leftovers before filling remaining slots
    rng.shuffle(leftovers)

    # --------------------------------------------------
    # 3) Fill val up to target size
    # --------------------------------------------------
    while len(train_patients) < target_train:
        train_patients.append(leftovers.pop())

    # --------------------------------------------------
    # 4) Fill test up to target size
    # --------------------------------------------------
    while len(test_patients) < target_test and leftovers:
        test_patients.append(leftovers.pop())

    # --------------------------------------------------
    # 5) Everything else goes to train
    # --------------------------------------------------
    val_patients.extend(leftovers)

    # --------------------------------------------------
    # 6) Rebalancing step
    #
    # If val or test became too large because of the constraints,
    # move random patients from these splits to train.
    #
    # Important:
    # This may slightly weaken the "at least 1 per group in val/test"
    # condition for very small edge cases, but usually keeps the split
    # much closer to the desired 70/15/15.
    # --------------------------------------------------
    if len(val_patients) > target_val:
        print(f"Val ist zu groß ({len(val_patients)} statt {target_val}), rebalancing...")
        rng.shuffle(val_patients)
        extra = len(val_patients) - target_val
        train_patients.extend(val_patients[:extra])
        val_patients = val_patients[extra:]

    if len(test_patients) > target_test:
        print(f"Test ist zu groß ({len(test_patients)} statt {target_test}), rebalancing...")
        rng.shuffle(test_patients)
        extra = len(test_patients) - target_test
        train_patients.extend(test_patients[:extra])
        test_patients = test_patients[extra:]

    # --------------------------------------------------
    # 7) If train is still too small, pull patients from the
    # larger split(s), but only if possible
    # --------------------------------------------------
    while len(train_patients) < target_train:
        if len(val_patients) > target_val:
            print(f"VAl ist zu klein ({len(val_patients)} statt {target_val}), rebalancing...")
            train_patients.append(val_patients.pop())
        elif len(test_patients) > target_test:
            print(f"Test ist zu klein ({len(test_patients)} statt {target_test}), rebalancing...")
            train_patients.append(test_patients.pop())
        else:
            # If no more balancing is possible, stop
            break

    # Final shuffle for nicer random order
    rng.shuffle(train_patients)
    rng.shuffle(val_patients)
    rng.shuffle(test_patients)

    return train_patients, val_patients, test_patients

In [54]:
def summarize_split(split_patients, records, name):
    """
    Print distribution of stratify groups inside one split.
    """
    rec_map = {r["patient_id"]: r for r in records}
    counts = Counter(rec_map[p]["stratify_label"] for p in split_patients)

    print(f"\n{name} ({len(split_patients)} patients)")
    for k, v in sorted(counts.items()):
        print(f"{k}: {v}")

In [55]:
train_patients, val_patients, test_patients = constrained_group_split(
    records,
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    random_state=42
)

print(f"Train: {len(train_patients)}")
print(f"Val:   {len(val_patients)}")
print(f"Test:  {len(test_patients)}")
print(train_patients)
summarize_split(train_patients, records, "TRAIN")
summarize_split(val_patients, records, "VAL")
summarize_split(test_patients, records, "TEST")

defaultdict(<class 'list'>, {})
defaultdict(<class 'list'>, {'A_only16': ['A0001', 'A0002', 'A0003', 'A0004', 'A0005', 'A0006', 'A0007', 'A0008', 'A0009', 'A0010', 'A0011', 'A0012', 'A0013', 'A0014', 'A0015', 'A0016', 'A0017', 'A0018', 'A0019', 'A0020', 'A0021', 'A0022', 'A0023', 'A0024', 'A0025', 'A0026', 'A0027', 'A0028', 'A0029', 'A0030', 'A0031', 'A0032', 'A0033', 'A0034', 'A0035', 'A0036', 'A0037', 'A0038', 'A0039', 'A0040', 'A0041', 'A0042', 'A0043', 'A0044', 'A0045', 'A0046', 'A0047', 'A0048', 'A0049', 'A0050', 'A0051', 'A0052', 'A0053', 'A0054', 'A0055', 'A0056', 'A0059', 'A0061', 'A0062', 'A0063', 'A0064', 'A0065', 'A0066', 'A0068', 'A0069', 'A0070', 'A0071', 'A0072', 'A0073', 'A0074', 'A0075', 'A0077', 'A0078', 'A0080', 'A0081', 'A0082', 'A0083', 'A0084', 'A0085', 'A0086', 'A0087', 'A0088', 'A0089', 'A0090', 'A0091', 'A0092', 'A0093', 'A0094', 'A0095', 'A0096', 'A0097', 'A0098', 'A0099', 'A0100', 'A0101', 'A0102', 'A0103', 'A0104', 'A0105', 'A0106', 'A0107', 'A0108', 'A0109',

In [56]:
# ----------------------------
# Helper functions
# ----------------------------

def apply_windowing(dicom_img, window_width=1400, window_level=-700):
    """
    Apply CT windowing to a DICOM image and return an 8-bit (0..255) image.

    - Converts raw pixel array to Hounsfield Units (HU) using slope/intercept (if present).
    - Clips values to [window_level - window_width/2, window_level + window_width/2].
    - Normalizes to 0..255 (uint8) for saving as PNG.
    """
    img = dicom_img.pixel_array
     # RGB / 8-bit nicht als CT fenstern
    if img.ndim == 3 and img.shape[-1] in (3, 4) and img.dtype == np.uint8:
        return img
    
    img = img.astype(np.float32)

    # Convert to HU if slope/intercept exist (common for CT)
    if 'RescaleIntercept' in dicom_img and 'RescaleSlope' in dicom_img:
        img = img * dicom_img.RescaleSlope + dicom_img.RescaleIntercept


    img_min = window_level - window_width // 2
    img_max = window_level + window_width // 2

    # Windowing (clip)
    windowed = np.clip(img, img_min, img_max)

    # Normalize to [0, 255] and cast to uint8
    return ((windowed - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)


def convert_to_yolo(size, box):
    """
    Convert bounding box coordinates from Pascal VOC format to YOLO format.

    Inputs:
      size: (width, height) of the image
      box:  [xmin, ymin, xmax, ymax] in absolute pixel coordinates

    Output:
      [x_center_norm, y_center_norm, width_norm, height_norm]
      where values are normalized to [0, 1].
    """
    dw = 1.0 / size[0]  # 1 / width
    dh = 1.0 / size[1]  # 1 / height

    # Center coordinates
    x = (box[0] + box[2]) / 2.0
    y = (box[1] + box[3]) / 2.0

    # Box width/height
    w = box[2] - box[0]
    h = box[3] - box[1]

    # Normalize
    return [x * dw, y * dh, w * dw, h * dh]

In [57]:
# ----------------------------
# Main export routine
# ----------------------------

def run_full_export(patient_list, split_name):
    """
    Export all images + annotations for the given patients into:
      EXPORT_BASE/split_name/images/*.png
      EXPORT_BASE/split_name/labels/*.txt

    Each DICOM slice that has a corresponding XML annotation (same SOPInstanceUID)
    is exported as:
      <patient>_<uid>.png  and  <patient>_<uid>.txt

    Returns:
      Number of exported images for that split.
    """
    img_count = 0
    missing_patient = 0
    unreadable_files = 0
    bit_counter = {} 

    for p_id in patient_list:
        bit_counter[p_id] = {"8bit": 0, "16bit": 0, "other": 0}
        # Folder with all XML files for this patient
        p_xml_dir = os.path.join(XML_ROOT, p_id)

        # Find a DICOM directory matching the patient id somewhere under DICOM_ROOT
        # (Assumes DICOM folder names contain p_id)
        p_dicom_search = glob.glob(os.path.join(DICOM_ROOT, f"*{p_id}*"))
        print(f"Patient {p_id}: Suche DICOMs in: {p_dicom_search}")
        if not p_dicom_search:
            print(f"⚠️  Warnung: Kein DICOM-Ordner gefunden für Patient {p_id}, überspringe...")
            missing_patient += 1
            continue
        # Collect all .dcm files recursively under that patient folder
        all_dicoms = glob.glob(os.path.join(p_dicom_search[0], "**", "*.dcm"), recursive=True)

        # Build a list of annotation ids (XML filenames without extension).
        # Convention: XML filename == SOPInstanceUID.xml
        xml_list = [
            f.replace(".xml", "")
            for f in os.listdir(p_xml_dir)
            if f.endswith(".xml")
        ]

        for d_path in all_dicoms:
            try:
                # Read only DICOM metadata first (fast) to get the SOPInstanceUID
                ds_meta = pydicom.dcmread(d_path, stop_before_pixels=True)
                uid = ds_meta.SOPInstanceUID

                # Only export if this DICOM slice has an annotation XML
                if uid in xml_list:
                    # Now read full pixel data
                    ds = pydicom.dcmread(d_path)

                    bits = getattr(ds, "BitsAllocated", None)
                    if bits == 8:
                        bit_counter[p_id]["8bit"] += 1
                        bit_tag = "8bit"
                    elif bits == 16:
                        bit_counter[p_id]["16bit"] += 1
                        bit_tag = "16bit"
                    else:
                        bit_counter[p_id]["other"] += 1
                        bit_tag = "other"


                    # Convert to 8-bit image using CT windowing
                    img = apply_windowing(ds)

                    # img auf 2D bringen, falls es (H,W,3) ist
                    if img.ndim == 3 and img.shape[-1] in (3, 4):
                        img = img[:, :, 0]  # oder img.mean(axis=-1)

                    # Image dimensions (note: img is 2D array => shape is (H, W))
                    h, w = img.shape

                    # Output base name includes patient id and SOPInstanceUID
                    file_base = f"{p_id}_{uid}_{bit_tag}"

                    # Output paths
                    img_out = os.path.join(EXPORT_BASE, split_name, 'images', f"{file_base}.png")
                    lbl_out = os.path.join(EXPORT_BASE, split_name, 'labels', f"{file_base}.txt")

                    # Save image as PNG
                    cv2.imwrite(img_out, img)

                    # Parse corresponding XML annotation file
                    tree = ET.parse(os.path.join(p_xml_dir, f"{uid}.xml"))

                    # Build YOLO label lines: "class x_center y_center width height"
                    # This code assumes a single class => class id 0 for all objects
                    yolo_lines = []
                    for obj in tree.getroot().findall('object'):
                        b = obj.find('bndbox')

                        # Read Pascal/VOC-style bounding box
                        box = [
                            float(b.find('xmin').text),
                            float(b.find('ymin').text),
                            float(b.find('xmax').text),
                            float(b.find('ymax').text)
                        ]

                        # Convert to YOLO normalized format
                        y_box = convert_to_yolo((w, h), box)

                        # Append line for YOLO txt file
                        yolo_lines.append(
                            f"0 {y_box[0]:.6f} {y_box[1]:.6f} {y_box[2]:.6f} {y_box[3]:.6f}"
                        )

                    # Write label file (one bbox per line)
                    with open(lbl_out, "w") as f:
                        f.write("\n".join(yolo_lines))
                    img_count += 1

            except Exception as e:
                print(f"⚠️  Fehler bei Datei: {d_path}")
                print(f"    Exception: {type(e).__name__}: {e}")
                print(getattr(ds_meta, "Modality", None), getattr(ds_meta, "NumberOfFrames", 1))
                print(img.shape)
                unreadable_files += 1
                continue


    # ----------------------------
    # image-level summary
    # ----------------------------
    total_8bit = sum(v["8bit"] for v in bit_counter.values())
    total_16bit = sum(v["16bit"] for v in bit_counter.values())
    total_other = sum(v["other"] for v in bit_counter.values())
    total_images = total_8bit + total_16bit + total_other

    image_level_summary = {
        "8bit": total_8bit,
        "16bit": total_16bit,
        "other": total_other,
        "total": total_images,
    }


     # ----------------------------
    # patient-level summary
    # ----------------------------
    patients_only_8bit = 0
    patients_only_16bit = 0
    patients_both = 0
    patients_other_only = 0
    patients_zero = 0
    
    for p_id, counts in bit_counter.items():
        n8 = counts["8bit"]
        n16 = counts["16bit"]
        no = counts["other"]

        if n8 == 0 and n16 == 0 and no == 0:
            patients_zero += 1
        elif n8 > 0 and n16 > 0:
            patients_both += 1
        elif n8 > 0 and n16 == 0 and no == 0:
            patients_only_8bit += 1
        elif n16 > 0 and n8 == 0 and no == 0:
            patients_only_16bit += 1
        else:
            patients_other_only += 1

    patient_level_summary = {
        "only_8bit": patients_only_8bit,
        "only_16bit": patients_only_16bit,
        "both_8bit_16bit": patients_both,
        "zero_images": patients_zero,
        "other_cases": patients_other_only,
        "total_patients": len(bit_counter),
    }


    print(f"Summary - Missing Patients: {missing_patient}, Unreadable Files: {unreadable_files}")
    print("Bit counter per patient:")
    for p_id, counts in bit_counter.items():
        print(p_id, counts)

    print("Image-level summary:", image_level_summary)
    print("Patient-level summary:", patient_level_summary)

    return {
        "img_count": img_count,
        "missing_patient": missing_patient,
        "unreadable_files": unreadable_files,
        "bit_counter": bit_counter,
        "image_level": image_level_summary,
        "patient_level": patient_level_summary,
    }

In [58]:

total_train = run_full_export(train_patients, 'train')


Patient A0031: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0031']
Patient A0231: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0231']
Patient G0029: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-G0029']
Patient A0260: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0260']
Patient A0069: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0069']
Patient A0225: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0225']
Patient B0007: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-B0007']
Patient B0022: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-B0022']
Patient G0038: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-G0038']
Patient A0182: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0182']
Patient A0261: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0261']
Patient B0033: Suche DICOMs in: ['D:/DIno-L

In [59]:
total_val = run_full_export(val_patients, 'val')


Patient A0020: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0020']
Patient A0046: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0046']
Patient A0097: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0097']
Patient A0078: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0078']
Patient A0052: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0052']
Patient A0095: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0095']
Patient A0084: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0084']
Patient G0019: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-G0019']
Patient A0203: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0203']
Patient A0208: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0208']
Patient A0099: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0099']
Patient G0055: Suche DICOMs in: ['D:/DIno-L

In [60]:
total_test = run_full_export(test_patients, 'test')


Patient A0176: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0176']
Patient A0007: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0007']
Patient A0130: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0130']
Patient G0005: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-G0005']
Patient A0101: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0101']
Patient A0022: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0022']
Patient A0235: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0235']
Patient G0033: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-G0033']
Patient A0038: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0038']
Patient A0135: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-A0135']
Patient G0056: Suche DICOMs in: ['D:/DIno-LungDet-Data/Lung-PET-CT-Dx\\Lung_Dx-G0056']
Patient G0004: Suche DICOMs in: ['D:/DIno-L

In [ ]:
from collections import defaultdict

def summarize_images(split_patients, bit_counter):
    """
    Summarize image counts for a patient split.

    Returns:
    - total image counts
    - counts by prefix/class (A, B, E, G)
    """
    summary = {
        "total": {"8bit": 0, "16bit": 0, "other": 0, "all": 0},
        "by_class": defaultdict(lambda: {"8bit": 0, "16bit": 0, "other": 0, "all": 0})
    }

    for p in split_patients:
        prefix = p[0]   # A, B, E, G
        counts = bit_counter[p]

        n8 = counts["8bit"]
        n16 = counts["16bit"]
        no = counts["other"]
        nall = n8 + n16 + no

        # total
        summary["total"]["8bit"] += n8
        summary["total"]["16bit"] += n16
        summary["total"]["other"] += no
        summary["total"]["all"] += nall

        # by class
        summary["by_class"][prefix]["8bit"] += n8
        summary["by_class"][prefix]["16bit"] += n16
        summary["by_class"][prefix]["other"] += no
        summary["by_class"][prefix]["all"] += nall

    return summary

In [66]:
train_summary = summarize_images(train_patients, total_train["bit_counter"])
val_summary = summarize_images(val_patients, total_val["bit_counter"])
test_summary = summarize_images(test_patients, total_test["bit_counter"])

In [67]:
def print_summary(name, summary):
    print(f"\n{name}")
    print("TOTAL:", summary["total"])

    print("BY CLASS:")
    for cls in ["A", "B", "E", "G"]:
        if cls in summary["by_class"]:
            print(cls, summary["by_class"][cls])
        else:
            print(cls, {"8bit": 0, "16bit": 0, "other": 0, "all": 0})

In [68]:
print_summary("TRAIN", train_summary)
print_summary("VAL", val_summary)
print_summary("TEST", test_summary)


TRAIN
TOTAL: {'8bit': 7423, '16bit': 14460, 'other': 0, 'all': 21883}
BY CLASS:
A {'8bit': 4411, '16bit': 9620, 'other': 0, 'all': 14031}
B {'8bit': 556, '16bit': 1300, 'other': 0, 'all': 1856}
E {'8bit': 0, '16bit': 79, 'other': 0, 'all': 79}
G {'8bit': 2456, '16bit': 3461, 'other': 0, 'all': 5917}

VAL
TOTAL: {'8bit': 1287, '16bit': 2819, 'other': 0, 'all': 4106}
BY CLASS:
A {'8bit': 941, '16bit': 2037, 'other': 0, 'all': 2978}
B {'8bit': 178, '16bit': 403, 'other': 0, 'all': 581}
E {'8bit': 0, '16bit': 43, 'other': 0, 'all': 43}
G {'8bit': 168, '16bit': 336, 'other': 0, 'all': 504}

TEST
TOTAL: {'8bit': 1517, '16bit': 2876, 'other': 0, 'all': 4393}
BY CLASS:
A {'8bit': 996, '16bit': 1926, 'other': 0, 'all': 2922}
B {'8bit': 212, '16bit': 385, 'other': 0, 'all': 597}
E {'8bit': 0, '16bit': 79, 'other': 0, 'all': 79}
G {'8bit': 309, '16bit': 486, 'other': 0, 'all': 795}
